# Create Reaches for RivMapper v2

Takes the per-river SWORD reaches produced by `01_sword_reach_extraction.ipynb`
and re-splits them into new RivMapper reaches whose lengths scale with
upstream drainage area. The motivation: SWORD's native ~10 km reach
segmentation is too fine for large rivers, where geomorphic reaches should
be tens of km long. v1 used the same approach against HydroRIVERS;
v2 ports the regression to SWORD's `facc` field (drainage area in km²,
same units as v1's `UPLAND_SKM`).

## Reach boundaries

Reach buffer polygons are cut by lines perpendicular to the *smoothed
valley direction* at each reach boundary — eliminating the overlap between
adjacent reach buffers that occurs with simple line-buffering on sinuous
rivers. The smoothed direction is computed by fitting a principal-axis line
through merged-channel points within an adaptive window (default
`1.5 × local reach length`), so the cut angle reflects the valley, not the
local channel curvature. The cuts also square off the most upstream and
most downstream ends, replacing the rounded buffer caps.

## Workflow per river

1. Load `{working_directory}\SWORD\Extracted_Rivers\{river_name}\{river_name}_SWORD_reaches.gpkg`
2. Sort by `reach_order` (1 = upstream) and reproject to local UTM
3. Merge SWORD reaches into a single continuous LineString
4. Walk downstream, sampling `facc` at each cumulative distance to compute
   the next reach length via `reach_len = modulator * SLOPE * facc + INTERCEPT`
5. Cut the merged line at the cumulative split distances
6. Drop zero-length artifacts and trim too-short final reaches
7. Compute a perpendicular cut line at each reach boundary (including river
   head and mouth) using the smoothed valley tangent
8. Buffer each reach line by `buffer_width * reach_len`, then clip with the
   adjacent cut lines to produce clean, non-overlapping polygons
9. Write `_reach_lines.gpkg` and `_reach_polygons.gpkg`

## Outputs per river

```
{working_directory}\RiverMapping\Reaches\{river_name}\
    ├── {river_name}_reach_lines.gpkg
    └── {river_name}_reach_polygons.gpkg
```

Both files carry the same five fields:

| field         | type   | description                                                   |
|---------------|--------|---------------------------------------------------------------|
| `ds_order`    | int    | reach order; 1 = most upstream                                |
| `reach_len`   | float  | reach length, meters (UTM)                                    |
| `ds_dist`     | float  | cumulative length of all upstream reaches, meters; US = 0     |
| `reach_ar`    | float  | clipped polygon area, km² (identical on lines and polygons)   |
| `us_drain_ar` | float  | SWORD `facc` at upstream end of the reach, km²                |

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import math
from pathlib import Path
from shapely.geometry import LineString, MultiLineString, Point
from shapely.ops import split

In [2]:
# ── CONFIG ───────────────────────────────────────────────────────────
CONFIG = {
    "master_csv_path": Path(
        r"E:\Dissertation\Data\Template_RivMapper_master.csv"
    ),
    # Empirical regression: reach_len_m = modulator * SLOPE * facc_km2 + INTERCEPT
    # Defaults are the v1 values (derived along the Ganges against
    # HydroRIVERS UPLAND_SKM, here applied to SWORD facc; both km²).
    "reach_len_slope": 0.0668,
    "reach_len_intercept": 14316.0,
    # If the final downstream reach is shorter than this fraction of the
    # second-to-last reach, drop it as an iterative-splitting artifact.
    "ds_trim_fraction": 0.8,
    # Tolerance for gaps between consecutive SWORD reaches when merging
    # into a single LineString (meters, UTM).
    "gap_tolerance_m": 10.0,
    # Smoothing window for computing the valley tangent at each reach
    # boundary. Window length = multiplier × local reach length.
    "smoothing_window_multiplier": 1.5,
    # Number of sample points along the merged line within each smoothing
    # window for the best-fit tangent.
    "n_tangent_samples": 21,
    # Cut line half-length = this × max per-reach buffer width.
    # Must be large enough that every cut fully crosses every buffer.
    "cut_extent_factor": 3.0,
    "overwrite": False,
}

print(f"Master CSV:                  {CONFIG['master_csv_path']}")
print(f"Reach-length slope:          {CONFIG['reach_len_slope']}")
print(f"Reach-length intercept:      {CONFIG['reach_len_intercept']}")
print(f"DS trim fraction:            {CONFIG['ds_trim_fraction']}")
print(f"Gap tolerance (m):           {CONFIG['gap_tolerance_m']}")
print(f"Smoothing window multiplier: {CONFIG['smoothing_window_multiplier']}")
print(f"Cut extent factor:           {CONFIG['cut_extent_factor']}")

Master CSV:                  E:\Dissertation\Data\Template_RivMapper_master.csv
Reach-length slope:          0.0668
Reach-length intercept:      14316.0
DS trim fraction:            0.8
Gap tolerance (m):           10.0
Smoothing window multiplier: 1.5
Cut extent factor:           3.0


In [3]:
def utm_epsg_for_gdf(gdf):
    """
    Return an "EPSG:xxxxx" string for the local UTM zone containing the
    centroid of `gdf`'s bounding box. Works whether the gdf is currently
    geographic or projected.
    """
    if gdf.crs is None or not gdf.crs.is_geographic:
        gdf_geo = gdf.to_crs("EPSG:4326")
    else:
        gdf_geo = gdf

    minx, miny, maxx, maxy = gdf_geo.total_bounds
    lon = (minx + maxx) / 2.0
    lat = (miny + maxy) / 2.0

    utm_zone = int((lon + 180.0) / 6.0) + 1
    epsg = (32600 if lat >= 0 else 32700) + utm_zone
    return f"EPSG:{epsg}"

In [4]:
def _merge_swords_to_line(swords_sorted, gap_tolerance_m=10.0):
    """
    Concatenate SWORD reaches (sorted upstream-to-downstream) into a single
    continuous LineString. Reach 1's orientation is set by comparing its
    endpoints against reach 2's endpoints; each subsequent reach is then
    reversed if its end is closer to the previous reach's end than its
    start is. Gaps larger than `gap_tolerance_m` print a warning.
    """
    if len(swords_sorted) == 0:
        return LineString()

    # Materialize all reach coord lists, collapsing any MultiLineStrings.
    coord_lists = []
    for _, row in swords_sorted.iterrows():
        geom = row.geometry
        if isinstance(geom, MultiLineString):
            geom = max(geom.geoms, key=lambda g: g.length)
        coord_lists.append(list(geom.coords))

    # Orient reach 1 by checking its endpoints against reach 2.
    if len(coord_lists) >= 2:
        s1, e1 = coord_lists[0][0], coord_lists[0][-1]
        s2, e2 = coord_lists[1][0], coord_lists[1][-1]
        d_e1 = min(math.dist(e1, s2), math.dist(e1, e2))
        d_s1 = min(math.dist(s1, s2), math.dist(s1, e2))
        if d_s1 < d_e1:
            coord_lists[0] = coord_lists[0][::-1]

    coords = list(coord_lists[0])
    prev_end = coords[-1]

    for i, pts in enumerate(coord_lists[1:], start=1):
        d_start = math.dist(pts[0], prev_end)
        d_end = math.dist(pts[-1], prev_end)
        if d_end < d_start:
            pts = pts[::-1]

        gap = math.dist(pts[0], prev_end)
        if gap > gap_tolerance_m:
            reach_id = swords_sorted.iloc[i]["reach_id"]
            reach_order = swords_sorted.iloc[i]["reach_order"]
            print(
                f"  Warning: {gap:.1f} m gap before reach_id {reach_id} "
                f"(reach_order {reach_order})."
            )
            coords.extend(pts)
        else:
            coords.extend(pts[1:])

        prev_end = coords[-1]

    return LineString(coords)

In [5]:
def _cut(line, distance):
    """Split a LineString at `distance` from its start; return both halves."""
    if distance <= 0.0 or distance >= line.length:
        return line, LineString()
    for i, coord in enumerate(line.coords):
        projected = line.project(Point(coord))
        if projected == distance:
            return (
                LineString(line.coords[: i + 1]),
                LineString(line.coords[i:]),
            )
        if projected > distance:
            cp = line.interpolate(distance)
            return (
                LineString(line.coords[:i] + [(cp.x, cp.y)]),
                LineString([(cp.x, cp.y)] + line.coords[i:]),
            )
    return line, LineString()


def _split_line_with_points(line, points):
    """Sequentially split `line` at each point in `points`."""
    segments = []
    current = line
    for p in points:
        d = current.project(p)
        seg, current = _cut(current, d)
        segments.append(seg)
    segments.append(current)
    return segments

In [6]:
def _sample_facc(distance_m, swords_with_cum):
    """
    Sample SWORD `facc` at a cumulative downstream distance (UTM meters).
    `swords_with_cum` must have `cum_start_m`, `cum_end_m`, and `facc`
    columns. Returns the facc of the SWORD reach containing the distance.
    """
    for _, reach in swords_with_cum.iterrows():
        if reach["cum_end_m"] >= distance_m:
            return float(reach["facc"])
    return float(swords_with_cum.iloc[-1]["facc"])


def _compute_split_distances(
    swords_with_cum, modulator, slope, intercept,
):
    """
    Walk from the upstream end of the merged line downstream, accumulating
    reach lengths whose values are determined by sampling `facc` at the
    current cumulative position. Returns a list of cumulative distances
    (meters) marking the upstream end of each new RivMapper reach.

    The first entry is always 0.0 (start of the river); the last entry
    is < total_length and marks the start of the final reach.
    """
    total_length_m = swords_with_cum["utm_len_m"].sum()
    cum_distances = [0.0]
    while True:
        facc = _sample_facc(cum_distances[-1], swords_with_cum)
        reach_len = modulator * slope * facc + intercept
        next_cum = cum_distances[-1] + reach_len
        if next_cum >= total_length_m:
            break
        cum_distances.append(next_cum)
    return cum_distances

In [7]:
def _smoothed_tangent_at_distance(
    merged_line, position_m, window_m, n_samples=21,
):
    """
    Compute the local valley tangent at `position_m` along `merged_line`
    by fitting a principal-axis line through samples within a window of
    `window_m` centered on that position.

    Returns a unit-vector (numpy 2-array) tangent that points downstream
    (in the direction of increasing position along the line). Smoothing
    is implicit in the line fit — high-frequency channel curvature
    averages out, leaving the valley direction.
    """
    total = merged_line.length
    half_window = window_m / 2.0
    start = max(0.0, position_m - half_window)
    end = min(total, position_m + half_window)
    if end - start < 1.0:
        # Degenerate window; fall back to a short local tangent.
        end = min(total, start + 1.0)

    distances = np.linspace(start, end, n_samples)
    points = [merged_line.interpolate(d) for d in distances]
    xs = np.array([p.x for p in points])
    ys = np.array([p.y for p in points])

    # Principal component via SVD of centered coords.
    cx = xs - xs.mean()
    cy = ys - ys.mean()
    M = np.column_stack([cx, cy])
    _, _, Vt = np.linalg.svd(M, full_matrices=False)
    tangent = Vt[0]  # leading direction, unit length

    # Sign-correct so the tangent points downstream.
    gross = np.array([xs[-1] - xs[0], ys[-1] - ys[0]])
    if np.dot(tangent, gross) < 0:
        tangent = -tangent
    return tangent


def _make_cut_line(point, tangent, half_length):
    """
    Construct a LineString perpendicular to `tangent` (2D unit vector),
    centered on `point`, extending `half_length` in each perpendicular
    direction.
    """
    perp = np.array([-tangent[1], tangent[0]])
    p1 = (point.x - perp[0] * half_length, point.y - perp[1] * half_length)
    p2 = (point.x + perp[0] * half_length, point.y + perp[1] * half_length)
    return LineString([p1, p2])


def _clip_polygon_by_cut(polygon, cut, anchor_point):
    """
    Split `polygon` by `cut` (a LineString that should fully cross the
    polygon) and return the piece that contains `anchor_point`. Falls
    back to the input polygon if the split fails or no piece contains
    the anchor.
    """
    if polygon.is_empty:
        return polygon
    try:
        result = split(polygon, cut)
    except Exception:
        return polygon

    pieces = list(result.geoms) if hasattr(result, "geoms") else [result]
    pieces = [p for p in pieces if p.geom_type in ("Polygon", "MultiPolygon")]
    if not pieces:
        return polygon

    for piece in pieces:
        if piece.contains(anchor_point):
            return piece
    # Anchor exactly on a boundary — return the nearest piece.
    return min(pieces, key=lambda p: p.distance(anchor_point))


def _compute_cut_lines(merged_line, reach_lens, ds_dists, config, max_buffer_m):
    """
    Build one cut line per reach boundary, including the river head
    (position 0) and mouth (position total_length). Returns a list of
    LineStrings of length len(reach_lens) + 1.

    Cut i is the upstream boundary of reach i; cut i+1 is the downstream
    boundary of reach i.
    """
    total_length_m = merged_line.length
    # Boundary positions: ds_dist values (one per reach, starting at 0)
    # plus the total length (downstream end of the last reach).
    positions = list(ds_dists) + [total_length_m]

    window_multiplier = config["smoothing_window_multiplier"]
    n_samples = config["n_tangent_samples"]
    cut_half_length = config["cut_extent_factor"] * max_buffer_m

    cut_lines = []
    n_reaches = len(reach_lens)
    for i, pos in enumerate(positions):
        if i == 0:
            local_len = reach_lens[0]
        elif i == n_reaches:  # final downstream cut
            local_len = reach_lens[-1]
        else:  # internal boundary between reaches i-1 and i
            local_len = max(reach_lens[i - 1], reach_lens[i])

        window_m = window_multiplier * local_len
        tangent = _smoothed_tangent_at_distance(
            merged_line, pos, window_m, n_samples,
        )
        cut_point = merged_line.interpolate(pos)
        cut_lines.append(_make_cut_line(cut_point, tangent, cut_half_length))

    return cut_lines

In [8]:
def process_river(row, config):
    """
    Re-split one river's extracted SWORD reaches into RivMapper reaches,
    buffer and clip them, and write both lines and polygons geopackages.

    Returns "ok (...)", "skipped", or "failed: <reason>".
    """
    river_name = row["river_name"]
    working_directory = Path(row["working_directory"])
    modulator = float(row["reach_length_modulator"])
    buffer_width = float(row["buffer_width"])

    # ── Input/output paths ──────────────────────────────────────────
    sword_reaches_path = (
        working_directory / "SWORD" / "Extracted_Rivers" / river_name
        / f"{river_name}_SWORD_reaches.gpkg"
    )
    out_dir = working_directory / "RiverMapping" / "Reaches" / river_name
    lines_out = out_dir / f"{river_name}_reach_lines.gpkg"
    polys_out = out_dir / f"{river_name}_reach_polygons.gpkg"

    if not sword_reaches_path.exists():
        return f"failed: SWORD reaches not found at {sword_reaches_path}"

    if not config["overwrite"] and lines_out.exists() and polys_out.exists():
        return "skipped"

    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Load extracted SWORD reaches ────────────────────────────────
    swords = gpd.read_file(sword_reaches_path, engine="pyogrio")
    if swords.empty:
        return "failed: SWORD reaches file is empty"
    if "reach_order" not in swords.columns:
        return "failed: SWORD reaches file missing `reach_order` column"

    swords = swords.sort_values("reach_order").reset_index(drop=True)

    # ── Reproject to local UTM for accurate length and area ─────────
    utm = utm_epsg_for_gdf(swords)
    swords = swords.to_crs(utm)

    swords["utm_len_m"] = swords.geometry.length
    swords["cum_end_m"] = swords["utm_len_m"].cumsum()
    swords["cum_start_m"] = swords["cum_end_m"] - swords["utm_len_m"]

    # ── Merge into single LineString ────────────────────────────────
    merged_line = _merge_swords_to_line(swords, config["gap_tolerance_m"])

    # ── Compute split distances via facc regression ─────────────────
    cum_distances = _compute_split_distances(
        swords,
        modulator=modulator,
        slope=config["reach_len_slope"],
        intercept=config["reach_len_intercept"],
    )
    cut_points = [merged_line.interpolate(d) for d in cum_distances[1:]]
    reach_geoms = _split_line_with_points(merged_line, cut_points)

    # ── Build the lines GeoDataFrame ────────────────────────────────
    reaches = gpd.GeoDataFrame(geometry=reach_geoms, crs=utm)
    reaches["reach_len"] = [g.length for g in reaches.geometry]
    reaches = reaches[reaches["reach_len"] > 0.0].reset_index(drop=True)
    if reaches.empty:
        return "failed: no non-empty reaches after splitting"

    # Trim the final downstream reach if much shorter than its upstream
    # neighbor.
    if len(reaches) >= 2:
        last_len = reaches.iloc[-1]["reach_len"]
        prev_len = reaches.iloc[-2]["reach_len"]
        if last_len < config["ds_trim_fraction"] * prev_len:
            reaches = reaches.iloc[:-1].copy()

    reaches = reaches.reset_index(drop=True)
    reaches["ds_order"] = range(1, len(reaches) + 1)

    upstream_cumlen = reaches["reach_len"].cumsum().shift(1, fill_value=0.0)
    reaches["ds_dist"] = upstream_cumlen

    reaches["us_drain_ar"] = [
        _sample_facc(d, swords) for d in reaches["ds_dist"]
    ]

    # ── Compute perpendicular cut lines at every boundary ──────────
    reach_lens = reaches["reach_len"].tolist()
    ds_dists = reaches["ds_dist"].tolist()
    max_buffer_m = float((reaches["reach_len"] * buffer_width).max())

    cut_lines = _compute_cut_lines(
        merged_line, reach_lens, ds_dists, config, max_buffer_m,
    )

    # ── Buffer each reach and clip with adjacent cut lines ─────────
    clipped_polygons = []
    for i in range(len(reaches)):
        reach_geom = reaches.iloc[i].geometry
        raw_buffer = reach_geom.buffer(reach_geom.length * buffer_width)
        midpoint = reach_geom.interpolate(reach_geom.length / 2.0)

        # cut_lines[i] = upstream cut; cut_lines[i+1] = downstream cut
        polygon = _clip_polygon_by_cut(raw_buffer, cut_lines[i], midpoint)
        polygon = _clip_polygon_by_cut(polygon, cut_lines[i + 1], midpoint)
        clipped_polygons.append(polygon)

    polygons = reaches.copy()
    polygons["geometry"] = clipped_polygons
    polygons["reach_ar"] = polygons.geometry.area / 1.0e6  # m² → km²

    reaches["reach_ar"] = polygons["reach_ar"].values

    field_order = ["ds_order", "reach_len", "ds_dist", "reach_ar", "us_drain_ar"]
    reaches = reaches[field_order + ["geometry"]]
    polygons = polygons[field_order + ["geometry"]]

    reaches.to_file(lines_out, driver="GPKG", engine="pyogrio")
    polygons.to_file(polys_out, driver="GPKG", engine="pyogrio")

    return f"ok ({len(reaches)} reaches)"

In [9]:
# ── Load and validate master CSV ────────────────────────────────────
master = pd.read_csv(CONFIG["master_csv_path"])
master = master.dropna(subset=["river_name"]).reset_index(drop=True)

required_cols = {
    "river_name", "working_directory",
    "reach_length_modulator", "buffer_width",
}
missing = required_cols - set(master.columns)
if missing:
    raise ValueError(f"Master CSV missing required columns: {missing}")

print(f"Loaded {len(master)} rivers from master CSV.\n")

# ── Process each river ──────────────────────────────────────────────
results = []
for _, row in master.iterrows():
    river_name = row["river_name"]
    print(f"→ {river_name} ... ", end="", flush=True)
    try:
        status = process_river(row, CONFIG)
    except Exception as e:
        status = f"failed: {type(e).__name__}: {e}"
    print(status)
    results.append({"river_name": river_name, "status": status})

# ── Summary ─────────────────────────────────────────────────────────
summary = pd.DataFrame(results)
print("\n──── Summary ────")
print(summary.to_string(index=False))

Loaded 2 rivers from master CSV.

→ OkTedi ... ok (11 reaches)
→ Beni ... ok (10 reaches)

──── Summary ────
river_name          status
    OkTedi ok (11 reaches)
      Beni ok (10 reaches)


## Notes

- **Cut algorithm.** At every reach boundary the tangent direction is
  computed by SVD on points sampled along the merged channel within a
  window of `smoothing_window_multiplier × local_reach_length` (default
  1.5). The boundary cut is perpendicular to that tangent, anchored at
  the channel position. This keeps the cut at the right longitudinal
  position along the river while averaging out meander noise.

- **End cuts.** The most upstream and most downstream reaches are also
  cut perpendicular (replacing the rounded buffer caps). The window at
  these ends is one-sided — sampling extends from the endpoint inward.

- **Adaptive window scales with reach length.** Because reach length
  itself scales with drainage area (via the facc regression), the
  smoothing window is automatically tighter for small upstream reaches
  and wider for large downstream reaches — the right behavior in both
  regimes.

- **Cut line length.** Cut lines are constructed to be
  `cut_extent_factor × max_buffer_width` long on each side of the
  channel (default 3×). If a cut ever fails to fully cross a buffer
  (extremely sinuous river, anomalously wide buffer), the clip falls
  back to the unclipped buffer and prints no error — visually inspect
  the polygons in QGIS to catch this case.

- **Pathologic edge cases.** On very sharp valley bends, adjacent cut
  lines may cross each other, producing thin sliver polygons or
  partially overlapping reach pieces. The current implementation does
  not detect this. If you see strange polygon shapes, the fix is to
  widen the smoothing window (raise `smoothing_window_multiplier`).

- **Regression coefficients ported from v1.** Defaults `SLOPE=0.0668`,
  `INTERCEPT=14316` were derived along the Ganges against HydroRIVERS
  `UPLAND_SKM`; SWORD `facc` is also km², so the formula is
  dimensionally consistent. Validate against a v1 output once before
  trusting the absolute reach counts.

## Next steps

- Validate visually in QGIS: confirm cuts are perpendicular to the
  valley, no overlap between adjacent reach polygons, square caps at
  river head and mouth.
- Consider detecting cut-line crossings and emitting a warning when
  they occur.